# The harness ladder

An agent is more than a model. Around every model sits a **harness**: the loop that calls it, the tools it is given, the messages it sees, and the rules that decide when it is finished. This notebook keeps the model the same and changes only the harness, one component at a time, to see how much the score changes.

The setup is kept small on purpose. Each of the ten tasks lives in its own folder under [`tasks/`](../tasks), for example `tasks/02_median/`, and contains three things:

- `repo/` is the small Python repository the agent works on: one module with one bug, one failing test in `repo/tests/`, and a file of sample data in `repo/data/`.
- `hidden/test_hidden.py` holds the **hidden tests** that grade the result. They never leave the task folder.
- `task.json` contains the bug report the agent receives, along with a known correct fix that only the lab's own tests read.

For every trial, `repo/` is copied into a temporary folder, and the agent works there with a single `bash` tool until it thinks the bug is fixed. Once it stops, the hidden tests run against that copy from outside it and decide whether the fix really works. The original task folder is never changed.

There are six versions, and each one adds a single component to the version before it:

| Version | Added component | The problem it addresses |
|---|---|---|
| v0 | a basic loop with only a bash tool | (the starting point) |
| v1 | a short `AGENTS.md` map | wasting steps looking around, and editing the tests |
| v2 | a check before "done" | saying "done" without running the tests |
| v3 | a loop detector and a limit on steps and cost | repeating the same command until the money runs out |
| v4 | short, cut-down tool output | getting lost in a very long output |
| v5 | a progress file | forgetting earlier work once it is out of view |

## Setup

Every run calls a real OpenAI model, so it needs `OPENAI_API_KEY` and it costs money. The key goes in a `.env` file at the root of the repository: copy `.env.example` to `.env` and paste the key after `OPENAI_API_KEY=`. Git ignores `.env`, so the key is never committed.

Three more settings control the run. Each can be set in `.env` or in the shell, and the shell takes precedence:

- `OPENAI_MODEL` chooses the model. The default is `gpt-6-luna`.
- `OPENAI_REASONING_EFFORT` sets how much the model reasons. The default is `low`.
- `HARNESS_TASKS` sets how many tasks run. The default is `3`, which comes to about 54 trials and about one dollar; `all` runs all ten.

Every run saves its trials, transcripts and chart in its own folder under `results/`.

The model's shell commands run on this machine, inside a temporary copy of each task and with a time limit, but without a sandbox.

In [1]:
import os
import textwrap
from dataclasses import dataclass, field
from pathlib import Path

import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from agent_harness_lab.grading import grade
from agent_harness_lab.metrics import summarize
from agent_harness_lab.models import DEFAULT_MODEL, REASONING_EFFORT, OpenAIClient, ToolCall
from agent_harness_lab.tasks import TASKS_DIR, Task, load_tasks, task_workspace
from agent_harness_lab.transcripts import save_transcript
from agent_harness_lab.workspace import changed_files, fingerprint, run_command, snapshot

N_TASKS = os.environ.get("HARNESS_TASKS", "3")
MODEL = DEFAULT_MODEL
TRIALS = 3

TASKS = load_tasks(limit=None if N_TASKS == "all" else int(N_TASKS))
RUN_DIR = TASKS_DIR.parent / "results" / f"{MODEL}_{len(TASKS)}tasks"
RUN_DIR.mkdir(parents=True, exist_ok=True)
print(f"model={MODEL} effort={REASONING_EFFORT} tasks={len(TASKS)} trials={TRIALS} -> {RUN_DIR}")

model=gpt-6-luna effort=low tasks=3 trials=3 -> /Users/linafaik/Documents/projects/agent-harness-lab/results/gpt-6-luna_3tasks


## The agent loop

The loop is the part of the harness that never changes. It sends the model a context, runs the command that comes back, and adds the output to the history. When the model replies without a command, it is saying the task is done.

Two details matter later:

- **The model sees only recent messages.** Besides the system prompt and the task, it sees only the last `WINDOW` messages, just as a real agent eventually runs out of room in its context.
- **The loop records which files each command changed.** It compares a snapshot of the workspace before and after each command. With that record, a component can tell whether the tests have run since the last edit, and the trace analysis can find edits to the tests.

Components connect to the loop at six points, called hooks. Each hook does nothing by default, so a component only defines the hooks it needs.

In [2]:
@dataclass
class TrialState:
    """Everything the loop and its components know about the trial so far."""

    task: Task
    workspace: Path
    steps: int = 0
    cost_usd: float = 0.0
    input_tokens: int = 0
    output_tokens: int = 0
    commands: list[str] = field(default_factory=list)
    events: list[dict] = field(default_factory=list)
    tests_ran_since_change: bool = False
    stop_reason: str = ""


class Component:
    """A harness component; every hook is a no-op until a subclass overrides it."""

    def system_prompt(self, state: TrialState) -> str:
        return ""  # Text appended to the system prompt once, at the start.

    def context_note(self, state: TrialState) -> str | None:
        return None  # A note pinned into the context at every step, never scrolled away.

    def on_output(self, output: str, state: TrialState) -> str:
        return output  # A chance to rewrite a tool output before the model reads it.

    def on_finish(self, state: TrialState) -> str | None:
        return None  # An objection that turns "done" back into another step.

    def after_step(self, state: TrialState) -> str | None:
        return None  # A message added to the history after a command has run.

    def should_stop(self, state: TrialState) -> str | None:
        return None  # A reason to end the trial before the model says it is done.

In [3]:
BASE_SYSTEM = (
    "You are a software engineer fixing a bug in a small Python repository. "
    "You have one tool, `bash`, which runs a shell command from the repository root. "
    "When the bug is fixed, reply with a short summary and no tool call."
)
MAX_STEPS = 30          # A safety cap shared by every version, not a design choice.
WINDOW = 12             # Recent messages visible besides the system prompt and the task.
COMMAND_TIMEOUT_S = 30


def recent_window(history: list[dict], size: int) -> list[dict]:
    """Returns the last `size` messages, never starting on an orphaned tool output."""
    window = history[-size:]
    while window and window[0]["role"] == "tool":
        window = window[1:]
    return window


def first(values):
    """Returns the first value that is not None, or None."""
    return next((value for value in values if value is not None), None)


def execute(call: ToolCall, state: TrialState, components: list[Component]) -> dict:
    """Runs one command, records what it changed, and returns the tool message."""
    before = snapshot(state.workspace)
    result = run_command(call.command, state.workspace, COMMAND_TIMEOUT_S)
    after = snapshot(state.workspace)
    changed = changed_files(before, after)
    if "pytest" in call.command:
        state.tests_ran_since_change = True
    elif changed:
        state.tests_ran_since_change = False

    output = f"exit code {result.exit_code}\n{result.output or '(no output)'}"
    for component in components:
        output = component.on_output(output, state)
    state.commands.append(call.command)
    state.events.append({
        "step": state.steps,
        "command": call.command,
        "exit_code": result.exit_code,
        "timed_out": result.timed_out,
        "raw_output_chars": len(result.output),
        "shown_output_chars": len(output),
        "changed_files": changed,
        "workspace": fingerprint(after),
    })
    return {"role": "tool", "tool_call_id": call.id, "content": output}


def run_agent(task: Task, workspace: Path, client, components: list[Component]) -> tuple[TrialState, list[dict]]:
    """Runs the agent on one task until it finishes or the harness stops it.

    Returns the final state and the full message history, which is longer than
    anything the model saw at a single step.
    """
    state = TrialState(task=task, workspace=workspace)
    system = {"role": "system", "content": BASE_SYSTEM + "".join(c.system_prompt(state) for c in components)}
    task_message = {"role": "user", "content": f"Fix this issue in the repository:\n\n{task.issue}"}
    history: list[dict] = []

    while True:
        reason = first(c.should_stop(state) for c in components)
        if reason or state.steps >= MAX_STEPS:
            state.stop_reason = reason or "max_steps"
            break
        notes = [{"role": "user", "content": note} for c in components if (note := c.context_note(state))]
        reply = client.complete([system, task_message, *notes, *recent_window(history, WINDOW)])
        state.steps += 1
        state.cost_usd += reply.cost_usd
        state.input_tokens += reply.input_tokens
        state.output_tokens += reply.output_tokens
        history.append(reply.message)

        if not reply.tool_calls:
            objection = first(c.on_finish(state) for c in components)
            if objection is None:
                state.stop_reason = "done"
                break
            history.append({"role": "user", "content": objection})
            continue

        for call in reply.tool_calls:
            history.append(execute(call, state, components))
        for component in components:
            if nudge := component.after_step(state):
                history.append({"role": "user", "content": nudge})

    return state, [system, task_message, *history]

## v1: a map of the repository

An `AGENTS.md` is a short guide file at the root of a repository, written for AI agents rather than for people: a README for the agent. In v1, the harness writes one into each task's copy and also pastes its text into the system prompt, so the agent has it from the very first step. It is built from the task's files; for the median task, it reads:

```
# AGENTS.md
- Source code: `stats.py`. Bugs are fixed there.
- Tests: `tests/`. They are the specification and must never be edited.
- Run the tests with `python -m pytest -q`.
```

The map addresses two problems:

- **Wasted steps.** Without it, the agent spends its first commands exploring (`ls -R`, `cat README.md`, `grep`) just to find the file with the bug. Each of those steps costs money and fills the short context window.
- **Unknown rules.** Nothing in v0 tells the agent that the tests are off-limits. An agent stuck on a failing test may "fix" the test instead of the code. The visible test then passes, but the hidden tests still fail, so the trial fails. The map states the rule explicitly.

The map appears in two places for a reason. The system prompt makes sure the agent reads it, and the file in the workspace lets the agent read it again later with `cat AGENTS.md`, just as it would with a real repository's guide file.

In [4]:
class AgentsMap(Component):
    """Writes a short AGENTS.md into the workspace and quotes it in the system prompt."""

    def system_prompt(self, state: TrialState) -> str:
        sources = ", ".join(f"`{p.name}`" for p in sorted(state.workspace.glob("*.py")))
        text = textwrap.dedent(f"""\
            # AGENTS.md
            - Source code: {sources}. Bugs are fixed there.
            - Tests: `tests/`. They are the specification and must never be edited.
            - Run the tests with `python -m pytest -q`.
        """)
        (state.workspace / "AGENTS.md").write_text(text)
        return f"\n\nThe repository map, also saved as AGENTS.md:\n\n{text}"

## v2: a check before "done"

Agents often say they are done right after an edit, because the edit *looks* right. This check turns that claim into a question the harness can answer: have the tests run since the last change to the workspace? If not, "done" is refused and the agent is asked to run them.

In [5]:
class TestGate(Component):
    """Refuses to finish until the tests have run since the last change."""

    def on_finish(self, state: TrialState) -> str | None:
        if state.tests_ran_since_change:
            return None
        return "You have not run the tests since your last change. Run the tests with `python -m pytest -q` before finishing."

## v3: a loop detector and a budget

Some failures do not end; they repeat. An agent can run the same command again and again, hoping each time for a different result. This version adds two safeguards:

- **The loop detector** notices when one command has run three times in a row without changing the workspace. It first warns the agent, and if the repetition continues, it ends the trial.
- **The budget** limits the number of steps and the dollars spent per trial. A trial that is going nowhere stops early instead of spending money until the safety limit.

In [6]:
class LoopDetector(Component):
    """Nudges, then stops, an agent that repeats a command with no effect on the workspace."""

    def __init__(self, repeats: int = 3):
        self.repeats = repeats
        self.warned_at: int | None = None

    def _looping(self, state: TrialState) -> bool:
        recent = state.events[-self.repeats:]
        return len(recent) == self.repeats and len({(e["command"], e["workspace"]) for e in recent}) == 1

    def after_step(self, state: TrialState) -> str | None:
        if self.warned_at is None and self._looping(state):
            self.warned_at = len(state.events)
            return "You are repeating the same command without new information. Step back, reconsider the failure and try a different approach."
        return None

    def should_stop(self, state: TrialState) -> str | None:
        repeated_since_warning = self.warned_at is not None and len(state.events) - self.warned_at >= self.repeats
        return "loop" if repeated_since_warning and self._looping(state) else None


class Budget(Component):
    """Ends the trial once it has used its step or cost allowance."""

    def __init__(self, max_steps: int = 20, max_cost_usd: float = 0.05):
        self.max_steps = max_steps
        self.max_cost_usd = max_cost_usd

    def should_stop(self, state: TrialState) -> str | None:
        if state.steps >= self.max_steps:
            return "step_budget"
        if state.cost_usd >= self.max_cost_usd:
            return "cost_budget"
        return None

## v4: short tool output

A single `cat` of a data file can put thousands of lines in front of the model. That costs money on every later step, because the output stays in the context, and it hides the few lines that matter. This component shortens every output before the model reads it:

- It removes blank lines.
- It keeps only the beginning and the end of long outputs.
- It adds a note saying how much was left out and how to see more.

In [7]:
class ConciseOutput(Component):
    """Strips blank lines and truncates long outputs to their head and tail."""

    def __init__(self, head_chars: int = 1200, tail_chars: int = 600):
        self.head_chars = head_chars
        self.tail_chars = tail_chars

    def on_output(self, output: str, state: TrialState) -> str:
        text = "\n".join(line.rstrip() for line in output.splitlines() if line.strip())
        if len(text) <= self.head_chars + self.tail_chars:
            return text
        omitted = len(text) - self.head_chars - self.tail_chars
        return (
            f"{text[:self.head_chars]}\n"
            f"[... {omitted} characters omitted; use head, tail or grep for a narrower view ...]\n"
            f"{text[-self.tail_chars:]}"
        )

## v5: a progress file

The context window is the agent's working memory, and it is short. Once an early attempt is out of view, the agent can no longer tell that it already tried that idea, and it may try it again. The progress file gives it a memory that stays in view: the agent writes one line per important action in `PROGRESS.md`, and the harness adds the file's content to the context at every step.

In [8]:
class ProgressFile(Component):
    """Asks the agent to keep PROGRESS.md and pins its content into every step."""

    def system_prompt(self, state: TrialState) -> str:
        return (
            "\n\nKeep a running log in PROGRESS.md: after each meaningful action, append one line "
            "saying what you did and what you learned. Its content is shown to you at every step, "
            "even after older messages have scrolled out of view."
        )

    def context_note(self, state: TrialState) -> str | None:
        path = state.workspace / "PROGRESS.md"
        notes = path.read_text().strip() if path.exists() else ""
        return f"PROGRESS.md, your notes so far:\n{notes[-2000:]}" if notes else None

## All six versions

Each version is a list of components, and each list is the previous one plus one new component. The list holds the classes rather than ready-made objects, so new components are created for every trial and nothing carries over from one trial to the next.

In [9]:
VERSIONS = {
    "v0": [],
    "v1": [AgentsMap],
    "v2": [AgentsMap, TestGate],
    "v3": [AgentsMap, TestGate, LoopDetector, Budget],
    "v4": [AgentsMap, TestGate, LoopDetector, Budget, ConciseOutput],
    "v5": [AgentsMap, TestGate, LoopDetector, Budget, ConciseOutput, ProgressFile],
}

## Running every version

Each trial goes through the same four steps:

1. The task's repository is copied into a new temporary folder.
2. The agent works on that copy until it stops or the harness stops it.
3. The hidden tests grade the copy the agent left behind.
4. The full transcript is saved, and one row is added to the results table.

The three trials of a task share nothing except the task itself.

In [10]:
def run_trial(version: str, task: Task, trial: int) -> dict:
    """Runs, grades and saves one trial, and returns its row for the results table."""
    with task_workspace(task) as workspace:
        client = OpenAIClient(MODEL)
        components = [make() for make in VERSIONS[version]]
        state, messages = run_agent(task, workspace, client, components)
        verdict = grade(task, workspace)
    row = {
        "version": version,
        "task": task.id,
        "trial": trial,
        "passed": verdict.passed,
        "stop_reason": state.stop_reason,
        "steps": state.steps,
        "cost_usd": round(state.cost_usd, 6),
        "input_tokens": state.input_tokens,
        "output_tokens": state.output_tokens,
    }
    save_transcript(RUN_DIR, {**row, "model": MODEL, "reasoning_effort": REASONING_EFFORT, "events": state.events,
                              "messages": messages, "grader_output": verdict.output})
    return row


rows = []
for version in VERSIONS:
    for task in TASKS:
        for trial in range(TRIALS):
            rows.append(run_trial(version, task, trial))
    done = [r for r in rows if r["version"] == version]
    print(f"{version}: {sum(r['passed'] for r in done)}/{len(done)} trials passed")

trials = pd.DataFrame(rows)
trials.to_csv(RUN_DIR / "trials.csv", index=False)

v0: 9/9 trials passed
v1: 9/9 trials passed
v2: 9/9 trials passed
v3: 9/9 trials passed
v4: 9/9 trials passed
v5: 9/9 trials passed


## Results

The summary reads from the saved `trials.csv`, so this part works the same on a new run or on earlier results. Four numbers describe each version:

- **pass@1** is the share of trials that passed, averaged over tasks. It is the chance that a single attempt succeeds.
- **pass^3** is stricter: the share of tasks on which all three trials passed. It measures how reliable the agent is rather than what it can do.
- **Cost** and **steps** are averaged over every trial.

In [11]:
trials = pd.read_csv(RUN_DIR / "trials.csv")
summary = summarize(trials)
summary.to_csv(RUN_DIR / "summary.csv")
summary

,pass_at_1,pass_hat_k,cost_per_task,steps_per_task,k
version,,,,,
v0,1.0,1.0,0.00053,4.44,3
v1,1.0,1.0,0.00034,4.11,3
v2,1.0,1.0,0.00031,3.89,3
v3,1.0,1.0,0.00032,4.00,3
v4,1.0,1.0,0.00033,4.00,3
v5,1.0,1.0,0.00047,4.22,3


In [12]:
SERIES = {"pass@1": "#2a78d6", "pass^3": "#eb6834"}
NEUTRAL, INK, MUTED, GRID = "#8a8984", "#0b0b0b", "#52514e", "#e8e7e3"


def ladder_figure(summary: pd.DataFrame, title: str) -> go.Figure:
    """Builds three small multiples: success rates, cost per task and steps per task."""
    versions = list(summary.index)
    fig = make_subplots(rows=1, cols=3, horizontal_spacing=0.08,
                        subplot_titles=("Success rate", "Cost per task (USD)", "Steps per task"))
    for name, column in (("pass@1", "pass_at_1"), ("pass^3", "pass_hat_k")):
        fig.add_trace(go.Scatter(
            x=versions, y=summary[column], name=name, mode="lines+markers",
            line=dict(color=SERIES[name], width=2), marker=dict(size=9),
            hovertemplate=f"{name} %{{x}}: %{{y:.0%}}<extra></extra>",
        ), row=1, col=1)
    fig.add_trace(go.Bar(x=versions, y=summary["cost_per_task"], marker_color=NEUTRAL, showlegend=False,
                         hovertemplate="%{x}: $%{y:.4f}<extra></extra>"), row=1, col=2)
    fig.add_trace(go.Bar(x=versions, y=summary["steps_per_task"], marker_color=NEUTRAL, showlegend=False,
                         hovertemplate="%{x}: %{y:.1f} steps<extra></extra>"), row=1, col=3)
    fig.update_yaxes(range=[0, 1.05], tickformat=".0%", row=1, col=1)
    fig.update_yaxes(rangemode="tozero", gridcolor=GRID, zeroline=False, color=MUTED)
    fig.update_xaxes(color=MUTED, showgrid=False)
    fig.update_layout(
        title=dict(text=title, font=dict(size=16, color=INK)), template="simple_white",
        font=dict(family="Helvetica, Arial, sans-serif", size=12, color=MUTED),
        legend=dict(orientation="h", y=-0.15, x=0), bargap=0.35,
        width=1100, height=420, margin=dict(t=80, b=60, l=50, r=20),
    )
    return fig


fig = ladder_figure(summary, f"Harness ladder: {MODEL} ({REASONING_EFFORT} effort), {len(TASKS)} tasks x {TRIALS} trials")
fig.write_html(RUN_DIR / "ladder.html", include_plotlyjs="cdn")
fig.write_json(RUN_DIR / "ladder.json")  # Rendered to PNG outside the kernel by scripts/export_chart.py.
fig

## Reading the traces

A score says *whether* the agent succeeded; the transcripts say *how*. The trace analysis flags three behaviours in every saved trial:

- edits to the test files;
- attempts to find the hidden tests outside the workspace;
- the same command run three or more times in a row.

The same script runs from the command line with `make analyze`.

In [13]:
import subprocess
import sys

print(subprocess.run([sys.executable, str(TASKS_DIR.parent / "scripts" / "analyze_traces.py"), str(RUN_DIR)],
                     capture_output=True, text=True).stdout)

Flagged trials per version:

         trials  passed  test_edit  grader_read  looping
version                                                 
v0            9       9          0            0        0
v1            9       9          0            0        0
v2            9       9          0            0        0
v3            9       9          0            0        0
v4            9       9          0            0        0
v5            9       9          0            0        0

